# Auto Loader Ingesta Incremental

# Auto Loader: ingesta incremental de archivos

**El problema:** cada día llegan archivos nuevos a la landing zone. Necesitamos cargar
**solo los nuevos**, exactamente una vez, aunque el job falle a la mitad y aunque el
esquema cambie con el tiempo. Con `spark.read` batch tendríamos que releer toda la
carpeta y llevar la cuenta a mano.

**La solución:** Auto Loader — la fuente `cloudFiles` de Structured Streaming. Escala a
miles de millones de archivos con garantía *exactly-once*, y recuerda por nosotros qué
archivos ya procesó.

En este notebook lo ejecutamos en **modo lote** con `Trigger.AvailableNow`: procesa
todo lo pendiente y termina. Es el patrón recomendado para cargas programadas. Y funciona en el free edition.

In [ ]:
%run ./60_Setup_Landing_Zone

La celda anterior reutiliza la configuración y las funciones generadoras del notebook 60

In [ ]:
# Limpieza local idempotente: este notebook resetea SU propio estado.
dbutils.fs.rm(LANDING_AL, recurse=True)
dbutils.fs.mkdirs(LANDING_AL)
dbutils.fs.rm(CHK_61, recurse=True)

spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.ratings_bronze_al")
print("Estado del notebook 41 reiniciado.")

In [ ]:
# El "sistema externo" deposita el lote 1 en la landing zone
generar_lote_ratings(LANDING_AL, lote=1, n=500)
display(dbutils.fs.ls(LANDING_AL))

## La sintaxis `cloudFiles`

* `.format("cloudFiles")` — activa Auto Loader como fuente del stream.
* `cloudFiles.format` — el formato de los archivos que llegan (`json`, `csv`, `parquet`, ...).
* `cloudFiles.schemaLocation` — dónde Auto Loader **guarda el esquema inferido** para no re-inferirlo
  en cada corrida y poder detectar cambios.
* `checkpointLocation` (en el `writeStream`) — dónde guarda **qué archivos ya procesó**
  (una base RocksDB embebida).

**El checkpoint es la memoria de la ingesta**: si se borra, Auto Loader olvida qué archivos vio
y los re-procesa todos. Por eso la celda de limpieza de arriba borra checkpoint + tabla juntos.

Sobre los tipos: por defecto, con JSON/CSV/XML Auto Loader infiere **todas las columnas como
`string`** (evita choques de tipos entre archivos). Aquí activamos
`cloudFiles.inferColumnTypes = true` para que infiera tipos reales; la alternativa quirúrgica
es `cloudFiles.schemaHints` para fijar el tipo de columnas puntuales. 
NOTA: no confundir con el `inferSchema` del lector batch que usamos en el notebook 30.

El lote 1 tiene 500 eventos. Después de ejecutar esta celda, ¿cuántas filas tendrá `ratings_bronze_al`? ¿Y qué tipo de dato tendría `rating` si NO activáramos `inferColumnTypes`: double o string?

In [ ]:
from pyspark.sql.functions import col, current_timestamp

def ingesta_autoloader():
    stream = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("cloudFiles.schemaLocation", f"{CHK_61}/schema")
              .option("cloudFiles.inferColumnTypes", "true")
              .load(LANDING_AL)
              .withColumn("_ingest_ts", current_timestamp())
              .withColumn("_source_file", col("_metadata.file_path"))  # input_file_name() está bloqueado en serverless
              .writeStream
              .option("checkpointLocation", f"{CHK_61}/chk")
              .option("mergeSchema", "true")   # la tabla Delta destino acepta columnas nuevas (lo usaremos al final)
              .trigger(availableNow=True)      # procesa lo pendiente y termina
              .toTable(f"{CATALOG}.{SCHEMA}.ratings_bronze_al"))
    stream.awaitTermination()                  # esperamos a que el lote termine antes de seguir

ingesta_autoloader()

tabla = spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al")
tabla.printSchema()
print("Filas:", tabla.count())

## `_rescued_data`: la red de seguridad

Además de las columnas del esquema, Auto Loader agrega automáticamente `_rescued_data`:
ahí cae todo dato que **no calza** con el esquema vigente — una columna que el esquema no
tiene, un tipo incompatible, o hasta una diferencia de mayúsculas. Guarda un blob JSON con
el dato rescatado y la ruta del archivo de origen, así **nunca se pierde información**
silenciosamente. En Silver es común revisarla como patrón de cuarentena.

En este primer lote todo calza, así que viene en `NULL`.

In [ ]:
display(spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al")
        .select("user_id", "movie_id", "rating", "ts", "_source_file", "_rescued_data")
        .limit(10))

Vamos a ejecutar EXACTAMENTE la misma ingesta otra vez, sin agregar archivos nuevos. ¿Cuántas filas tendrá la tabla al terminar: 500 o 1000?

In [ ]:
ingesta_autoloader()
print("Filas después de re-ejecutar:", spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al").count())

**500.** El checkpoint recuerda que `lote_01.json` ya fue procesado: no hay nada nuevo que
ingerir. Esa es la garantía *exactly-once* — re-ejecutar es seguro y no duplica datos.

Contraste con `spark.read` batch: habría releído toda la carpeta y un `append` habría
duplicado las 500 filas.

In [ ]:
# Llega el lote 2 (500 eventos más)
generar_lote_ratings(LANDING_AL, lote=2, n=500)
display(dbutils.fs.ls(LANDING_AL))

Ahora sí hay un archivo nuevo en el landing zon. ¿Cuántas filas procesará esta corrida: 0, 500 o 1000? ¿Y cuántas tendrá la tabla al final?

In [ ]:
ingesta_autoloader()

tabla = spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al")
print("Filas totales:", tabla.count())

# Equé archivo aportó qué
display(tabla.groupBy("_source_file").count().orderBy("_source_file"))

## Evolución de esquema

¿Qué pasa cuando el sistema externo empieza a mandar una **columna nueva**? Auto Loader lo
controla con `cloudFiles.schemaEvolutionMode`:

| Modo | Comportamiento |
|---|---|
| `addNewColumns` *(default con inferencia)* | El stream **falla** con `UnknownFieldException`, actualiza el esquema guardado en `schemaLocation`, y al reiniciar continúa con la columna nueva. |
| `rescue` | El esquema **no** evoluciona; lo nuevo cae en `_rescued_data`. |
| `failOnNewColumns` | Falla y **no** actualiza; requiere intervención manual. |
| `none` | Ignora las columnas nuevas. |

Que el default sea "fallar" es intencional: un cambio de esquema es un evento que alguien
debe ver. En producción se combina con **Lakeflow Jobs**, que reinicia el stream
automáticamente; aquí haremos el reinicio a mano.

In [ ]:
# El lote 3 trae una columna nueva: device
generar_lote_ratings(LANDING_AL, lote=3, n=500, columnas_extra=True)
display(dbutils.fs.ls(LANDING_AL))

El lote 3 trae una columna nueva (`device`). Con el modo por defecto (`addNewColumns`), ¿la próxima ingesta va a: (a) ignorar la columna, (b) fallar y requerir reinicio, o (c) agregarla silenciosamente?

In [ ]:
try:
    ingesta_autoloader()
    print("La corrida terminó sin detectar columnas nuevas (no era lo esperado en este punto).")
except Exception as e:
    print("Auto Loader detectó la columna nueva 'device':")
    print("  1. Actualizó el esquema guardado en schemaLocation.")
    print("  2. Detuvo el stream (UnknownFieldException) — esto es intencional, no un bug.")
    print()
    print("En producción, Lakeflow Jobs reinicia el stream automáticamente.")
    print("Aquí lo reintentamos a mano:")
    print()
    ingesta_autoloader()

tabla = spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al")
tabla.printSchema()   # ahora incluye device
print("Filas totales:", tabla.count())

Los lotes viejos no tienen valor para `device`, así que quedan en `NULL`
(gracias al `mergeSchema` del `writeStream`, la tabla Delta aceptó la columna nueva):

In [ ]:
display(spark.table(f"{CATALOG}.{SCHEMA}.ratings_bronze_al")
        .groupBy("_source_file", "device")
        .count()
        .orderBy("_source_file", "device"))

## Cierre: qué recuerda cada pieza

| Pieza | Qué recuerda |
|---|---|
| `checkpointLocation` | Qué **archivos** ya fueron procesados (exactly-once). |
| `cloudFiles.schemaLocation` | El **esquema** vigente y su historial de evolución. |
| Tabla Delta | Los **datos** (y acepta columnas nuevas con `mergeSchema`). |

Auto Loader tiene dos modos de descubrimiento de archivos —
*directory listing* (default: lista el directorio; sin permisos extra) y *file notification*
(se suscribe a la cola de eventos de la nube). En Free Edition solo aplica directory listing.